# RAG Document Question-Answering Assistant

Ask questions about PDF documents. Built with **LangChain, FAISS and Hugging Face**.

**Pipeline:** PDFs -> overlapping chunks -> sentence-transformer embeddings -> FAISS index -> retrieve top-k chunks -> Hugging Face question-answering model -> answer with source file and page number.

Run all cells top to bottom (Colab: Runtime > Run all). Every number printed below was measured in this run.

## Setup

In [ ]:
!pip -q install langchain-community langchain-text-splitters langchain-huggingface faiss-cpu pypdf sentence-transformers transformers
!mkdir -p pdfs
!wget -q -O pdfs/attention.pdf https://arxiv.org/pdf/1706.03762
!wget -q -O pdfs/bert.pdf https://arxiv.org/pdf/1810.04805

## Building blocks

In [ ]:
import glob, json, os, time
import numpy as np
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import pipeline

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
QA_MODEL = "deepset/roberta-base-squad2"

def build_index(pdf_dir, chunk_size=800, overlap=150):
    docs = []
    for p in sorted(glob.glob(os.path.join(pdf_dir, "*.pdf"))):
        docs += PyPDFLoader(p).load()          # one Document per page; metadata has source + page (0-based)
    chunks = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap).split_documents(docs)
    t0 = time.perf_counter()
    store = FAISS.from_documents(chunks, HuggingFaceEmbeddings(model_name=EMBED_MODEL))
    return store, len(docs), len(chunks), time.perf_counter() - t0

def answer(store, qa, question, k=4):
    best = None
    for doc, _ in store.similarity_search_with_score(question, k=k):
        out = qa(question=question, context=doc.page_content)
        if best is None or out["score"] > best["score"]:
            best = {"answer": out["answer"], "confidence": round(float(out["score"]), 3),
                    "source": os.path.basename(doc.metadata.get("source", "")), "page": int(doc.metadata.get("page", 0)) + 1}
    return best

def evaluate(store, qa, items, k=4):
    hit1 = hitk = ans_ok = n_ans = 0
    lat_ret, lat_total = [], []
    for it in items:
        t0 = time.perf_counter(); hits = store.similarity_search_with_score(it["question"], k=k); lat_ret.append((time.perf_counter() - t0) * 1000)
        locs = [(os.path.basename(d.metadata.get("source", "")), int(d.metadata.get("page", 0)) + 1) for d, _ in hits]
        want = (it["pdf"], int(it["page"]))
        hit1 += locs[0] == want; hitk += want in locs
        t0 = time.perf_counter(); res = answer(store, qa, it["question"], k); lat_total.append((time.perf_counter() - t0) * 1000)
        if it.get("answer_contains"):
            n_ans += 1; ans_ok += it["answer_contains"].lower() in res["answer"].lower()
    n = len(items)
    out = {"questions": n, "k": k, "retrieval_hit_at_1": round(hit1 / n, 3), f"retrieval_hit_at_{k}": round(hitk / n, 3),
           "avg_retrieval_ms": round(float(np.mean(lat_ret)), 1), "avg_end_to_end_ms": round(float(np.mean(lat_total)), 1)}
    if n_ans: out["answer_contains_accuracy"] = round(ans_ok / n_ans, 3)
    return out

## Build the index and ask a question

In [ ]:
PDF_DIR = "pdfs"          # downloaded above; upload your own PDFs into this folder to use them instead
os.makedirs("outputs", exist_ok=True)
store, n_pages, n_chunks, t_index = build_index(PDF_DIR)
qa = pipeline("question-answering", model=QA_MODEL)
rag_results = {"embedding_model": EMBED_MODEL, "qa_model": QA_MODEL, "pdf_pages": n_pages, "chunks": n_chunks, "index_build_s": round(t_index, 1)}

question = "What is the Transformer?"
rag_results["example"] = {"question": question, **answer(store, qa, question)}
print(rag_results["example"])

## Retrieval accuracy (fill in your own checked questions)

In [ ]:
# OPTIONAL but recommended: measure retrieval accuracy.
# Open the PDFs yourself, write 10-20 questions whose answers you have CHECKED, and fill them in, e.g.
# {"question": "...", "pdf": "attention.pdf", "page": 3, "answer_contains": "short phrase from the answer"}
EVAL_ITEMS = []

if EVAL_ITEMS:
    rag_results["evaluation"] = evaluate(store, qa, EVAL_ITEMS, k=4)
print(json.dumps(rag_results, indent=2))
json.dump(rag_results, open("outputs/rag_results.json", "w"), indent=2)